# R2-U2 - Protocol B frozen AST head warm-up

Run this notebook in Google Colab with a GPU from **your own writable Drive project copy**. Set `PROJECT_ROOT` yourself. The notebook writes checkpoints, a ledger row and evidence reports into that copy; use a private copy rather than the teammates' shared source.

The task trains AST's `classifier.*` (including classifier normalization) while the pretrained backbone stays frozen and in evaluation mode. It uses B training recordings and selection validation only. Calibration, operating and final-test audio are excluded. The WavLM cache, fixed crop bank and BiGRU statistics are not inputs.

The predeclared recipe is in `configs/r2_u2_ast_b.json`: two epochs, head AdamW `1e-4`, weight decay `0.01`, constant schedule, microbatch four with accumulation four (effective batch 16), evaluation batch 16 and FP16. The guide distinguishes task requirements, R1 evidence and newly recorded baseline decisions. No cells in this delivered notebook have been executed; no package installation is performed.

Phases: configure -> runtime/source checks -> metadata preflight -> fixture/integration tests -> stage required shards -> train or resume -> independent artifact verification -> handoff. See `docs/r2_u2_ast_warmup.md`.


In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, platform, subprocess, re, uuid, importlib.util, shutil
from datetime import datetime, timezone
from google.colab import drive

drive.mount("/content/drive")
PROJECT_ROOT = ""  # REQUIRED: absolute path to YOUR OWN writable Drive project copy
RUN_NAME = "r2_u2_ast_b_baseline_v1"  # keep this name to resume the same run
RESUME = False  # set True deliberately after an interrupted/completed run; rerun all cells

assert PROJECT_ROOT, "Set PROJECT_ROOT to your own project copy."
ROOT = Path(PROJECT_ROOT).resolve()
assert ROOT.is_absolute() and (ROOT / "pyproject.toml").is_file(), ROOT
assert re.fullmatch(r"r2_u2_[A-Za-z0-9_-]+", RUN_NAME), "Use a simple r2_u2_ run name."
RUN_DIR = ROOT / "runs/r2" / RUN_NAME
SCRATCH = Path("/content/echoproof-r2-u2").resolve()
assert SCRATCH.parent == Path("/content") and SCRATCH != ROOT
SCRATCH.mkdir(parents=True, exist_ok=True)
(SCRATCH / "tmp").mkdir(exist_ok=True)
os.environ.update(PYTHONDONTWRITEBYTECODE="1", TMPDIR=str(SCRATCH / "tmp"),
                  TMP=str(SCRATCH / "tmp"), TEMP=str(SCRATCH / "tmp"),
                  CUBLAS_WORKSPACE_CONFIG=":4096:8")
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "src"))
REPORTS = ROOT / "reports/r2"
REPORTS.mkdir(parents=True, exist_ok=True)
TESTS_PASSED = False
TRAINING_VERIFIED = False
INTEGRITY_PASSED = False
TEST_PATH = None
print(json.dumps({"project": str(ROOT), "run_directory": str(RUN_DIR),
                  "resume": RESUME, "scratch": str(SCRATCH)}, indent=2))


## 1. Check the prepared runtime and source files

Choose a GPU runtime. Missing imports or an incompatible runtime stop execution; prepare the environment separately and restart before retrying. Transformers 5.22.0 is the recorded AST model-library version. The R1 AST probe used PyTorch 2.11.0+cu130, while the later R2-U1 run used 2.14.1+cu130. Neither historical run establishes compatibility for this session: the AST/trainer tests below must pass without skips.

This cell parses the source and notebook code without executing research audio or downloading the model.


In [ ]:
TESTS_PASSED = TRAINING_VERIFIED = INTEGRITY_PASSED = False
import ast
syntax_files = list((ROOT / "src/echoproof").rglob("*.py")) + [
    ROOT / "scripts/r2_u2_ast_warmup.py",
    ROOT / "tests/test_r2_ast_warmup.py",
    ROOT / "tests/test_r1_u2_ast.py",
    ROOT / "tests/test_m1_trainer.py"]
for source_path in syntax_files:
    ast.parse(source_path.read_text(encoding="utf-8-sig"), filename=str(source_path))
notebook = json.loads((ROOT / "notebooks/R2_U2_Colab_AST_Warmup.ipynb").read_text(encoding="utf-8"))
for index, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] == "code":
        ast.parse("".join(cell["source"]), filename=f"R2_U2_Colab_AST_Warmup.ipynb:cell{index}")
required = ["numpy", "pandas", "pyarrow", "scipy", "sklearn", "soundfile", "librosa", "yaml",
            "torch", "torchaudio", "transformers", "huggingface_hub", "safetensors", "pytest"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
assert not missing, f"Missing dependencies: {missing}. Prepare the environment and restart; nothing was installed."
import torch, torchaudio, transformers
assert torch.cuda.is_available(), "Select a GPU runtime, restart, and rerun."
assert transformers.__version__ == "5.22.0", f"Expected transformers==5.22.0; found {transformers.__version__}"
import echoproof
assert Path(echoproof.__file__).resolve().is_relative_to(ROOT), "Another EchoProof installation shadows this project."
from echoproof.training import r2_u2
from echoproof.artifacts import write_json, file_digest
SOURCE_HASHES = r2_u2.source_hashes(ROOT)
ENVIRONMENT = {"python": platform.python_version(), "torch": torch.__version__,
               "torchaudio": torchaudio.__version__, "transformers": transformers.__version__,
               "gpu": torch.cuda.get_device_name(0),
               "gpu_vram_bytes": torch.cuda.get_device_properties(0).total_memory}
print(json.dumps(ENVIRONMENT, indent=2))


## 2. Validate metadata and inspect the frozen recipe

Preflight verifies the configured input hashes, split and leakage evidence, prior AST/trainer evidence, speaker boundaries, allowed components and complete canonical selection windows. It reads metadata only. It does not resample data, regenerate splits/crops, read research audio or fetch weights.

Review the printed recipe before training. Changing the recipe after a run starts requires a separate declared run; resume rejects incompatible source, input, configuration and runtime identities.


In [ ]:
RECIPE = r2_u2.load_recipe(ROOT)
PLAN = r2_u2.preflight(ROOT, RECIPE)
assert r2_u2.source_hashes(ROOT) == SOURCE_HASHES, "Source/config changed; restart from the top."
print(json.dumps({"recipe_sha256": file_digest(ROOT / "configs/r2_u2_ast_b.json"),
                  "train": RECIPE["train"], "decisions": RECIPE["decisions"],
                  "checkpoint_policy": RECIPE["checkpoint_policy"],
                  "plan_sha256": PLAN.sha256, "counts": PLAN.counts,
                  "required_shards": PLAN.shards()}, indent=2))


## 3. Pass fixture and AST/trainer integration tests

The tests use generated fixtures and tiny models. They establish software behavior, including head-only updates and frozen evaluation mode, checkpoint reload and recovery; they do not establish real-data accuracy. The notebook records the actual test output and source fingerprints. Any failed or skipped test blocks research-audio staging and training.

Acceptance-test skips and FP16 optimizer-step skips are different. The former block this gate; the latter are recorded by the trainer and assessed alongside the actual training/update checks.


In [ ]:
TESTS_PASSED = TRAINING_VERIFIED = INTEGRITY_PASSED = False
assert r2_u2.source_hashes(ROOT) == SOURCE_HASHES, "Source/config changed; restart from the top."
test_command = [sys.executable, "-B", "-m", "pytest", "tests/test_r2_ast_warmup.py",
                "tests/test_r1_u2_ast.py", "tests/test_m1_trainer.py",
                "-q", "-ra", "-p", "no:cacheprovider"]
test_env = dict(os.environ, PYTHONPATH=str(ROOT / "src"), PYTHONDONTWRITEBYTECODE="1")
started = time.perf_counter()
completed = subprocess.run(test_command, cwd=ROOT, env=test_env, text=True, capture_output=True)
print(completed.stdout)
print(completed.stderr)
output = completed.stdout + "\n" + completed.stderr
passed = re.findall(r"(\d+) passed", output)
skipped = bool(re.search(r"\b\d+ skipped\b|\bSKIPPED\b", output))
TEST_REPORT = {"task": "R2-U2 fixture verification", "command": test_command,
               "returncode": completed.returncode, "passed_count": int(passed[-1]) if passed else 0,
               "skipped": skipped, "stdout": completed.stdout, "stderr": completed.stderr,
               "summary": output.strip().splitlines()[-1] if output.strip() else "",
               "seconds": round(time.perf_counter() - started, 3),
               "source_sha256": SOURCE_HASHES, "environment": ENVIRONMENT}
TEST_PATH = REPORTS / ("r2_u2_tests_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
                       + "_" + uuid.uuid4().hex[:8] + ".json")
write_json(TEST_PATH, TEST_REPORT)
assert r2_u2.source_hashes(ROOT) == SOURCE_HASHES, "Source/config changed during tests."
TESTS_PASSED = completed.returncode == 0 and TEST_REPORT["passed_count"] > 0 and not skipped
assert TESTS_PASSED, f"Tests did not pass without skips; inspect {TEST_PATH}. Research audio has not been read."
print("Software gate passed. Real AST warm-up, checkpoint checks and teammate review remain pending.")


## 4. Stage only the plan's canonical shards on Colab scratch

The raw `dataset/` is unnecessary. Shards can physically contain excluded recordings; the guarded loader opens only the permitted training and selection members. Copying a shard is not authorization to use its other recordings.

Each staged file is checked with streaming SHA-256. A differing existing scratch file stops execution. This cell does not change persistent canonical input files.


In [ ]:
assert TESTS_PASSED and r2_u2.source_hashes(ROOT) == SOURCE_HASHES
SHARDS = SCRATCH / "shards"
SHARDS.mkdir(exist_ok=True)
required_bytes = sum((ROOT / "data/processed/cohort_v1" / name).stat().st_size for name in PLAN.shards())
additional_bytes = sum((ROOT / "data/processed/cohort_v1" / name).stat().st_size
                       for name in PLAN.shards() if not (SHARDS / name).exists())
free_bytes = shutil.disk_usage(SCRATCH).free
assert free_bytes > additional_bytes, "Insufficient scratch space for the required shard copies."
print(json.dumps({"required_shard_bytes": required_bytes, "additional_copy_bytes": additional_bytes,
                  "scratch_free_bytes_before_copy": free_bytes,
                  "note": "Model retrieval and training also require additional scratch/persistent space."}, indent=2))
STAGING = []
stage_start = time.perf_counter()
for name in PLAN.shards():
    assert re.fullmatch(r"shard_\d+\.tar", name) and Path(name).name == name, name
    source = ROOT / "data/processed/cohort_v1" / name
    destination = SHARDS / name
    assert source.is_file(), f"Missing canonical shard: {source}"
    if destination.exists():
        source_hash = file_digest(source)
        assert file_digest(destination) == source_hash, f"Scratch copy differs: {destination}; inspect before retrying."
        reused = True
    else:
        temporary = SHARDS / ("." + name + "." + uuid.uuid4().hex + ".partial")
        hasher = hashlib.sha256()
        try:
            with source.open("rb") as src, temporary.open("xb") as dst:
                for block in iter(lambda: src.read(1024 * 1024), b""):
                    hasher.update(block)
                    dst.write(block)
                dst.flush()
                os.fsync(dst.fileno())
            source_hash = hasher.hexdigest()
            assert file_digest(temporary) == source_hash, f"Copy checksum failed: {name}"
            assert not destination.exists(), f"Another writer created {destination}"
            temporary.replace(destination)
        finally:
            temporary.unlink(missing_ok=True)
        reused = False
    STAGING.append({"file": name, "sha256": source_hash, "bytes": destination.stat().st_size, "reused": reused})
    print(name, "reused" if reused else "copied", flush=True)
STAGING_SECONDS = round(time.perf_counter() - stage_start, 3)
print(json.dumps({"shards": len(STAGING), "bytes": sum(s["bytes"] for s in STAGING),
                  "seconds": STAGING_SECONDS}, indent=2))


## 5. Train or resume the declared warm-up

The runner retrieves the pinned AST checkpoint/processor, checks the loading audit and preprocessing, and uses the shared trainer. It trains only `classifier.*`; the backbone stays in evaluation mode. Publication requires the original backbone to remain unchanged, the head to change, finite saved results, complete selection predictions, checkpoint reload consistency and a matching ledger row.

For a fresh run use `RESUME=False`. To recover the same run, set `RESUME=True` in the first cell and rerun all cells. Keep `RUN_NAME` unchanged. A completed run can be reverified with resume; it must not create another training trial or duplicate completion ledger row.

A hard termination can leave `.writer.lock`. Confirm that the previous writer has stopped before manually removing **that lock only** in your own output copy, then resume. This notebook never removes a writer lock automatically.


In [ ]:
TRAINING_VERIFIED = INTEGRITY_PASSED = False
assert TESTS_PASSED and TEST_PATH is not None
assert r2_u2.source_hashes(ROOT) == SOURCE_HASHES, "Source/config changed; restart from the top."
assert not (RUN_DIR / ".writer.lock").exists(), (
    "Writer lock exists. Confirm the old writer stopped before manually removing that lock in your own copy.")
if not RESUME:
    assert not RUN_DIR.exists() or not any(RUN_DIR.iterdir()), (
        "Run directory has existing files. Use RESUME=True for this run, or deliberately choose a new declared run.")
train_before = set(REPORTS.glob("r2_u2_train_*.json"))
command = [sys.executable, "-B", str(ROOT / "scripts/r2_u2_ast_warmup.py"), "train",
           "--root", str(ROOT), "--run-dir", str(RUN_DIR), "--shard-dir", str(SHARDS),
           "--model-cache", str(SCRATCH / "model-cache"), "--test-report", str(TEST_PATH)]
if RESUME:
    command.append("--resume")
train_result = subprocess.run(command, cwd=ROOT, env=test_env)
new_train_reports = set(REPORTS.glob("r2_u2_train_*.json")) - train_before
assert len(new_train_reports) == 1, "Expected one new train evidence report; inspect reports/r2."
TRAIN_PATH = new_train_reports.pop()
TRAIN_REPORT = json.loads(TRAIN_PATH.read_text(encoding="utf-8"))
assert train_result.returncode == 0, f"Training/publication failed; inspect {TRAIN_PATH} before resuming."
assert TRAIN_REPORT.get("verified") is True, f"GPU/training/publication gates did not pass: {TRAIN_PATH}"
MANIFEST_PATH = RUN_DIR / "r2_u2_manifest.json"
assert MANIFEST_PATH.is_file(), "Required handoff manifest is absent."
TRAINING_VERIFIED = True
print("Warm-up evidence:", TRAIN_PATH)
print("Frozen comparison / R3 model initializer:", RUN_DIR / "best.pt")
print("Warm-up recovery state:", RUN_DIR / "latest.pt")


## 6. Independently verify persistent artifacts and publish the handoff

The verifier checks the committed artifacts and provenance without another model-training run. The final report records the real evidence links and exact checkpoint paths. It records success pending Muneeb's review; it does not record his approval or claim that R3 partial adaptation has happened.

Keep `best.pt` as the frozen-AST comparison and R3-U2 model initializer. Keep `latest.pt` for R2-U2 recovery. R3 changes the permitted trainable layers and builds a new optimizer; it does not continue the warm-up optimizer blindly.


In [ ]:
INTEGRITY_PASSED = False
assert TESTS_PASSED and TRAINING_VERIFIED
assert r2_u2.source_hashes(ROOT) == SOURCE_HASHES, "Source/config changed; do not accept this handoff."
verify_before = set(REPORTS.glob("r2_u2_verify_*.json"))
verify_command = [sys.executable, "-B", str(ROOT / "scripts/r2_u2_ast_warmup.py"), "verify",
                  "--root", str(ROOT), "--run-dir", str(RUN_DIR)]
verify_result = subprocess.run(verify_command, cwd=ROOT, env=test_env)
new_verify_reports = set(REPORTS.glob("r2_u2_verify_*.json")) - verify_before
assert len(new_verify_reports) == 1, "Expected one new integrity-verification report."
VERIFY_PATH = new_verify_reports.pop()
VERIFY_REPORT = json.loads(VERIFY_PATH.read_text(encoding="utf-8"))
assert verify_result.returncode == 0, f"Artifact verification failed: {VERIFY_PATH}"
assert VERIFY_REPORT.get("integrity_passed") is True, f"Integrity gate did not pass: {VERIFY_PATH}"
assert r2_u2.source_hashes(ROOT) == SOURCE_HASHES, "Source/config changed during verification."
INTEGRITY_PASSED = True
FINAL_REPORT = {"task": "R2-U2", "created_utc": datetime.now(timezone.utc).isoformat(),
                "status": "colab_verified_pending_muneeb_review", "verified": True,
                "environment": ENVIRONMENT, "source_sha256": SOURCE_HASHES,
                "tests": {"file": TEST_PATH.relative_to(ROOT).as_posix(), "sha256": file_digest(TEST_PATH)},
                "training": {"file": TRAIN_PATH.relative_to(ROOT).as_posix(), "sha256": file_digest(TRAIN_PATH)},
                "independent_verification": {"file": VERIFY_PATH.relative_to(ROOT).as_posix(), "sha256": file_digest(VERIFY_PATH)},
                "manifest": {"file": MANIFEST_PATH.relative_to(ROOT).as_posix(), "sha256": file_digest(MANIFEST_PATH)},
                "run_directory": RUN_DIR.relative_to(ROOT).as_posix(),
                "frozen_comparison_and_r3_initializer": (RUN_DIR / "best.pt").relative_to(ROOT).as_posix(),
                "warmup_resume_checkpoint": (RUN_DIR / "latest.pt").relative_to(ROOT).as_posix(),
                "ledger_file": "reports/ledger.csv",
                "staging": {"seconds": STAGING_SECONDS, "shards": STAGING},
                "scope": "Protocol B training and selection validation only",
                "class_mapping": {"0": "genuine", "1": "tts"},
                "independent_reviewer": None, "partial_adaptation": "pending R3-U2"}
FINAL_PATH = REPORTS / ("r2_u2_colab_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
                        + "_" + uuid.uuid4().hex[:8] + ".json")
write_json(FINAL_PATH, FINAL_REPORT)
print(json.dumps({"status": FINAL_REPORT["status"], "report": str(FINAL_PATH),
                  "manifest": str(MANIFEST_PATH), "frozen_comparison_and_r3_initializer": str(RUN_DIR / "best.pt"),
                  "warmup_resume_checkpoint": str(RUN_DIR / "latest.pt"),
                  "ledger": str(ROOT / "reports/ledger.csv")}, indent=2))
